# 데이터 전처리 결과서 생성

`churn_prediction_model.ipynb` 의 전처리 과정을 그대로 재현하여, 각 피처별 통계와 처리 방법을 **`reports/preprocessing_report.csv`** 로 저장한다.

- 1행 = 1피처
- 헤더·설명 모두 한글
- 사용 피처 18개 + 라벨/제외 피처도 참고용으로 포함
- Excel 한글 호환을 위해 `utf-8-sig` 로 저장

In [1]:
import pandas as pd
import numpy as np
from functools import reduce
from scipy.stats import linregress
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로드 (churn_prediction_model.ipynb 와 동일)

In [2]:
wide_df = pd.read_csv('../../data/raw/filtered_dataset_wide.csv')
long_df = pd.read_csv('../../data/raw/filtered_dataset_long.csv')
view_volatility_df = pd.read_csv('../../data/raw/view_volatility_output.csv')
upload_regularity_df = pd.read_csv('../../data/raw/upload_regularity_score.csv')
active_viewer_df = pd.read_csv('../../data/raw/active_viewer_score.csv')
sensitive_keyword_df = pd.read_csv('../../data/raw/sensitive_keyword_score.csv')

print(f'Wide : {wide_df.shape}')
print(f'Long : {long_df.shape}')

Wide : (3336, 22)
Long : (166449, 18)


## 2. 라벨링 (180일 이상 미업로드 = 이탈)

In [3]:
CHURN_THRESHOLD = 180
wide_df['is_churned'] = (wide_df['days_since_last_upload'] >= CHURN_THRESHOLD).astype(int)
print(wide_df['is_churned'].value_counts())
print(f"이탈 비율: {wide_df['is_churned'].mean():.2%}")

is_churned
0    2641
1     695
Name: count, dtype: int64
이탈 비율: 20.83%


## 3. 시계열 파생 피처 추출 (LONG_FEATURES)

In [4]:
long_df['published_at'] = pd.to_datetime(long_df['published_at'])
NOW = long_df['published_at'].max()

def extract_long_features(group):
    g = group.sort_values('published_at')
    g['ym'] = g['published_at'].dt.to_period('M')
    monthly = g.groupby('ym').size().reset_index(name='cnt')
    if len(monthly) >= 3:
        slope, *_ = linregress(range(len(monthly)), monthly['cnt'])
    else:
        slope = np.nan

    t3 = NOW - pd.Timedelta(days=90)
    recent_3m = (g['published_at'] >= t3).sum()

    views = g['view_count'].dropna().tolist()
    if len(views) >= 10:
        recent_avg = np.mean(views[-10:])
        total_avg  = np.mean(views)
        view_ratio = round(recent_avg / (total_avg + 1e-9), 4)
    else:
        view_ratio = np.nan

    dates = g['published_at'].dropna().tolist()
    if len(dates) >= 2:
        days_last2 = (dates[-1] - dates[-2]).days
    else:
        days_last2 = np.nan

    return pd.Series({
        'upload_slope': slope,
        'recent_3m_upload_count': int(recent_3m),
        'recent_view_ratio': view_ratio,
        'days_between_last2': days_last2,
    })

long_features = long_df.groupby('channel_id').apply(extract_long_features).reset_index()
print(long_features.shape)

(3336, 5)


## 4. 병합 (wide + long_features + 4개 보조 CSV, channel_id inner join)

In [5]:
df = wide_df.merge(long_features, on='channel_id', how='left')

view_volatility_df = view_volatility_df[['channel_id', 'volatility_prob']]
upload_regularity_df = upload_regularity_df[['channel_id', 'regularity_score']]
active_viewer_df = active_viewer_df[['channel_id', 'active_viewer_score', 'view_per_sub']]
sensitive_keyword_df = sensitive_keyword_df[['channel_id', 'sensitive_score']]

dfs = [df, view_volatility_df, upload_regularity_df, active_viewer_df, sensitive_keyword_df]
merged_df = reduce(lambda l, r: pd.merge(l, r, on='channel_id', how='inner'), dfs)
print(f'merged_df shape: {merged_df.shape}')

merged_df shape: (3333, 32)


## 5. 피처 메타정보 정의 (한글 설명·그룹·소스·비고)

In [6]:
WIDE_FEATURES = [
    'subscriber_count', 'view_count', 'total_video_count',
    'avg_upload_interval_days', 'std_upload_interval_days', 'max_gap_days', 'hiatus_count_30d',
    'avg_view_count', 'std_view_count', 'avg_like_count', 'avg_comment_count', 'avg_engagement_rate',
    'shorts_ratio', 'avg_shorts_view', 'avg_normal_view'
]
LONG_FEATURES = ['upload_slope', 'recent_view_ratio', 'days_between_last2']
ALL_FEATURES = WIDE_FEATURES + LONG_FEATURES

# 결과서에 함께 기록할 라벨/제외 피처
EXTRA_FEATURES = [
    'is_churned', 'days_since_last_upload',
    'recent_3m_upload_count',
    'volatility_prob', 'regularity_score',
    'active_viewer_score', 'view_per_sub', 'sensitive_score'
]

FEATURE_META = {
    # WIDE — 채널규모
    'subscriber_count':         ('구독자 수',                       '채널규모',     'filtered_dataset_wide.csv',     ''),
    'view_count':               ('채널 누적 조회수',                 '채널규모',     'filtered_dataset_wide.csv',     ''),
    'total_video_count':        ('채널 전체 영상 수',                '채널규모',     'filtered_dataset_wide.csv',     ''),
    # WIDE — 업로드패턴
    'avg_upload_interval_days': ('평균 업로드 간격(일)',             '업로드패턴',   'filtered_dataset_wide.csv',     ''),
    'std_upload_interval_days': ('업로드 간격 표준편차',             '업로드패턴',   'filtered_dataset_wide.csv',     '들쭉날쭉함 측정'),
    'max_gap_days':             ('최대 업로드 공백 기간(일)',         '업로드패턴',   'filtered_dataset_wide.csv',     ''),
    'hiatus_count_30d':         ('30일 이상 쉰 횟수',                '업로드패턴',   'filtered_dataset_wide.csv',     '휴지기 빈도'),
    # WIDE — 성과·참여도
    'avg_view_count':           ('영상당 평균 조회수',                '성과·참여도',  'filtered_dataset_wide.csv',     ''),
    'std_view_count':           ('영상당 조회수 변동성',              '성과·참여도',  'filtered_dataset_wide.csv',     ''),
    'avg_like_count':           ('영상당 평균 좋아요 수',              '성과·참여도',  'filtered_dataset_wide.csv',     ''),
    'avg_comment_count':        ('영상당 평균 댓글 수',                '성과·참여도',  'filtered_dataset_wide.csv',     ''),
    'avg_engagement_rate':      ('평균 참여율((좋아요+댓글)/조회수)',   '성과·참여도',  'filtered_dataset_wide.csv',     '분모 0 케이스 극단값 가능'),
    # WIDE — 쇼츠전략
    'shorts_ratio':             ('전체 영상 중 쇼츠 비중',             '쇼츠전략',     'filtered_dataset_wide.csv',     ''),
    'avg_shorts_view':          ('쇼츠 영상 평균 조회수',              '쇼츠전략',     'filtered_dataset_wide.csv',     ''),
    'avg_normal_view':          ('일반 영상 평균 조회수',              '쇼츠전략',     'filtered_dataset_wide.csv',     ''),
    # LONG 파생
    'upload_slope':             ('월별 업로드 수 선형회귀 기울기',     '시계열파생',   'filtered_dataset_long.csv(파생)', '음수=활동 감소'),
    'recent_view_ratio':        ('최근 10개 평균 조회수 / 전체 평균',  '시계열파생',   'filtered_dataset_long.csv(파생)', '1 미만이면 최근 성과 하락'),
    'days_between_last2':       ('마지막 두 영상 간격(일)',            '시계열파생',   'filtered_dataset_long.csv(파생)', ''),
    # 라벨 / 제외
    'is_churned':               ('이탈 라벨(180일 이상 미업로드)',     '라벨',         'filtered_dataset_wide.csv(파생)', '타깃 변수'),
    'days_since_last_upload':   ('마지막 업로드 이후 경과일',           '제외피처',     'filtered_dataset_wide.csv',     '데이터 리크: 라벨 생성에만 사용'),
    'recent_3m_upload_count':   ('최근 90일 업로드 수',                '제외피처',     'filtered_dataset_long.csv(파생)', '노트북에서 주석처리(미사용)'),
    'volatility_prob':          ('조회수 변동성 확률',                  '제외피처',     'view_volatility_output.csv',     '머지만 하고 학습 미사용'),
    'regularity_score':         ('업로드 규칙성 점수',                  '제외피처',     'upload_regularity_score.csv',    '머지만 하고 학습 미사용'),
    'active_viewer_score':      ('활성 시청자 점수',                    '제외피처',     'active_viewer_score.csv',        '머지만 하고 학습 미사용'),
    'view_per_sub':             ('구독자 대비 조회수 비율',             '제외피처',     'active_viewer_score.csv',        '머지만 하고 학습 미사용'),
    'sensitive_score':          ('민감 키워드 점수',                    '제외피처',     'sensitive_keyword_score.csv',    '머지만 하고 학습 미사용'),
}

assert set(ALL_FEATURES + EXTRA_FEATURES) <= set(FEATURE_META.keys()), '메타정보 누락 피처 존재'

## 6. 결측치 처리 (median 대체) — 처리 전 통계 기록 → 처리 → 후 통계 기록

In [7]:
n_rows = len(merged_df)

# 처리 전: 결측치 개수 기록
missing_before = {f: int(merged_df[f].isna().sum()) for f in ALL_FEATURES + EXTRA_FEATURES if f in merged_df.columns}

# 사용 피처에 대해서만 median 대체 (churn_prediction_model.ipynb와 동일)
fill_values = {}
for feat in ALL_FEATURES:
    med = merged_df[feat].median()
    fill_values[feat] = med
    merged_df[feat] = merged_df[feat].fillna(med)

# 처리 후 검증
assert merged_df[ALL_FEATURES].isna().sum().sum() == 0, '전처리 후에도 결측치 존재'
print('전처리 후 ALL_FEATURES 결측치 합:', merged_df[ALL_FEATURES].isna().sum().sum())

전처리 후 ALL_FEATURES 결측치 합: 0


## 7. 결과서 빌드

In [8]:
def fmt(v):
    if pd.isna(v):
        return ''
    if isinstance(v, (int, np.integer)):
        return int(v)
    return round(float(v), 4)

rows = []
for feat in ALL_FEATURES + EXTRA_FEATURES:
    if feat not in merged_df.columns:
        continue
    kor, group, src, note = FEATURE_META[feat]
    series = merged_df[feat]
    miss = missing_before.get(feat, int(series.isna().sum()))
    used = feat in ALL_FEATURES

    if used:
        fill_method = 'median'
        fill_val = fmt(fill_values[feat])
    elif feat == 'is_churned':
        fill_method = '없음'
        fill_val = ''
    elif feat == 'days_since_last_upload':
        fill_method = '라벨생성용(미처리)'
        fill_val = ''
    else:
        fill_method = '미적용(학습 미사용)'
        fill_val = ''

    rows.append({
        '피처명': feat,
        '한글설명': kor,
        '피처그룹': group,
        '소스파일': src,
        '데이터타입': str(series.dtype),
        '행수': n_rows,
        '결측치수': miss,
        '결측치비율': f'{miss / n_rows * 100:.2f}%',
        '결측치처리방법': fill_method,
        '대체값': fill_val,
        '최솟값': fmt(series.min()),
        '최댓값': fmt(series.max()),
        '평균': fmt(series.mean()),
        '표준편차': fmt(series.std()),
        '중앙값': fmt(series.median()),
        '모델사용여부': used,
        '비고': note,
    })

report_df = pd.DataFrame(rows)
print(f'결과서 shape: {report_df.shape}')
report_df

결과서 shape: (26, 17)


,피처명,한글설명,피처그룹,소스파일,데이터타입,행수,결측치수,결측치비율,결측치처리방법,대체값,최솟값,최댓값,평균,표준편차,중앙값,모델사용여부,비고
0,subscriber_count,구독자 수,채널규모,filtered_dataset_wide.csv,int64,3333,0,0.00%,median,283000.0,98900.0000,1.490000e+06,3.565883e+05,2.299918e+05,2.830000e+05,True,
1,view_count,채널 누적 조회수,채널규모,filtered_dataset_wide.csv,int64,3333,0,0.00%,median,116641786.0,0.0000,5.648210e+09,2.068600e+08,2.755631e+08,1.166418e+08,True,
2,total_video_count,채널 전체 영상 수,채널규모,filtered_dataset_wide.csv,int64,3333,0,0.00%,median,606.0,16.0000,2.589900e+04,1.223177e+03,1.938812e+03,6.060000e+02,True,
3,avg_upload_interval_days,평균 업로드 간격(일),업로드패턴,filtered_dataset_wide.csv,float64,3333,1,0.03%,median,3.59,0.0000,1.080000e+02,8.278000e+00,1.154240e+01,3.590000e+00,True,
4,std_upload_interval_days,업로드 간격 표준편차,업로드패턴,filtered_dataset_wide.csv,float64,3333,1,0.03%,median,3.705,0.0000,4.612000e+02,1.838500e+01,3.673600e+01,3.705000e+00,True,들쭉날쭉함 측정
5,max_gap_days,최대 업로드 공백 기간(일),업로드패턴,filtered_dataset_wide.csv,float64,3333,0,0.00%,median,18.0,0.0000,3.231000e+03,1.088881e+02,2.373893e+02,1.800000e+01,True,
6,hiatus_count_30d,30일 이상 쉰 횟수,업로드패턴,filtered_dataset_wide.csv,float64,3333,0,0.00%,median,0.0,0.0000,2.700000e+01,2.168300e+00,3.958300e+00,0.000000e+00,True,휴지기 빈도
7,avg_view_count,영상당 평균 조회수,성과·참여도,filtered_dataset_wide.csv,float64,3333,0,0.00%,median,74718.28,42.8600,3.016950e+07,1.915166e+05,6.386713e+05,7.471828e+04,True,
8,std_view_count,영상당 조회수 변동성,성과·참여도,filtered_dataset_wide.csv,float64,3333,0,0.00%,median,92205.96,28.6000,1.042189e+08,3.568742e+05,2.054044e+06,9.220596e+04,True,
9,avg_like_count,영상당 평균 좋아요 수,성과·참여도,filtered_dataset_wide.csv,float64,3333,66,1.98%,median,1053.3,0.4000,2.142704e+05,2.806225e+03,6.488024e+03,1.053300e+03,True,


## 8. CSV 저장 (`reports/preprocessing_report.csv`)

In [9]:
out_path = Path('../../reports/preprocessing_report.csv').resolve()
out_path.parent.mkdir(parents=True, exist_ok=True)
report_df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {out_path}')
print(f'행수: {len(report_df)} / 컬럼수: {len(report_df.columns)}')

저장 완료: /Volumes/Jexists/gitHub/skn_project/SKN30-2nd-1Team/reports/preprocessing_report.csv
행수: 26 / 컬럼수: 17
